Cell 1-Mount Google Drive

In [ ]:
# Connect Google Drive to Colab so we can access our project folder
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
## Cell 2 — Find your folder
import os

# This prints everything in your Drive root — find your project folder name here
os.listdir('/content/drive/My Drive/')

['GDToT',
 'GAURI_CV.gdoc',
 'Adhaar.png',
 'IMG_8352.jpeg',
 'Aadhar.pdf',
 'IMG_8355.jpeg',
 'Colab Notebooks',
 'Information Sheet.pdf',
 'CustomerBehaviourProject']

In [ ]:
## Cell 3 - Import libraries
import pandas as pd
import numpy as np
print('Libraries loaded successfully')
print(f'Pandas version: {pd.__version__}')

Libraries loaded successfully
Pandas version: 2.2.2


In [ ]:
# encoding='latin-1' → needed because UK product names have special characters
# without this Python throws a UnicodeDecodeError
FILE_PATH = '/content/drive/My Drive/CustomerBehaviourProject/Dataset/online_retail_II.csv'
df = pd.read_csv(FILE_PATH, encoding='latin-1')
print(f'Loaded successfully!')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

Loaded successfully!
Shape: 1,067,371 rows x 8 columns


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
print('======= COLUMN TYPES =======')
print(df.dtypes)

print('\n======= MISSING VALUES =======')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({
    'missing count': missing,
    'missing %': missing_pct
})[missing > 0])

print('\n======= BASIC STATS =======')
df[['Quantity', 'Price']].describe()

======= COLUMN TYPES =======
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

======= MISSING VALUES =======
             missing count  missing %
Description           4382       0.41
Customer ID         243007      22.77

======= BASIC STATS =======


,Quantity,Price
count,1.067371e+06,1.067371e+06
mean,9.938898e+00,4.649388e+00
std,1.727058e+02,1.235531e+02
min,-8.099500e+04,-5.359436e+04
25%,1.000000e+00,1.250000e+00
50%,3.000000e+00,2.100000e+00
75%,1.000000e+01,4.150000e+00
max,8.099500e+04,3.897000e+04


In [ ]:
original_count = len(df)
print(f'Original row count saved: {original_count:,}')

Original row count saved: 1,067,371


In [ ]:
## standardise column names
###**Why:** SQL and Python both prefer `snake_case`. Spaces in column names cause errors in SQL queries.
###This converts `Customer ID` → `customer_id`, `Invoice Date` → `invoice_date`, etc.
df.columns = (
    df.columns
    .str.strip()        # remove accidental spaces
    .str.lower()        # all lowercase
    .str.replace(' ', '_')  # spaces to underscores
)
print('Column names after cleaning:')
print(df.columns.tolist())

Column names after cleaning:
['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country']


In [ ]:
## Cell 8 - Remove cancelled orders
#**Why:** Invoices starting with `'C'` are cancellations. They have negative quantities.
#If we keep them, our revenue totals will be wrong and customer spend will be understated.
#In corporate analytics, cancelled orders are usually analysed separately not mixed into sales data.
cancelled_mask = df['invoice'].astype(str).str.startswith('C')
cancelled_count = cancelled_mask.sum()

print(f'Cancelled orders found: {cancelled_count:,} ({cancelled_count/len(df)*100:.1f}% of data)')

df = df[~cancelled_mask]
print(f'Rows remaining: {len(df):,}')


Cancelled orders found: 19,494 (1.8% of data)
Rows remaining: 1,047,877


In [ ]:
##Removing Rows with no customer ID because if we dont have it we can't consider it in customer's Journey.
missing_cust = df['customer_id'].isnull().sum()
print(f'Rows missing Customer ID: {missing_cust:,} ({missing_cust/len(df)*100:.1f}%)')

df = df.dropna(subset=['customer_id'])
print(f'Rows remaining: {len(df):,}')

Rows missing Customer ID: 242,257 (23.1%)
Rows remaining: 805,620


In [ ]:
# quantity or price of zero or below has no business meaning in sales data
# likely data entry errors or internal test records
# & means AND → both conditions must pass for a row to be kept
bad_qty   = (df['quantity'] <= 0).sum()
bad_price = (df['price'] <= 0).sum()

print(f'Rows with quantity <= 0: {bad_qty:,}')
print(f'Rows with price <= 0:    {bad_price:,}')

df = df[(df['quantity'] > 0) & (df['price'] > 0)]
print(f'Rows remaining: {len(df):,}')

Rows with quantity <= 0: 0
Rows with price <= 0:    71
Rows remaining: 805,549


In [ ]:
# invoicedate loaded as a string - convert to datetime so we can group by month,
#                                   calculate recency, and do any time-based analysis
# customer_id loaded as float (because of earlier missing values) - convert to int
#                                   for cleaner SQL queries and joins
df['invoicedate'] = pd.to_datetime(df['invoicedate'])
df['customer_id'] = df['customer_id'].astype(int)

print('Data types after fixing:')
print(df.dtypes)

Data types after fixing:
invoice                object
stockcode              object
description            object
quantity                int64
invoicedate    datetime64[ns]
price                 float64
customer_id             int64
country                object
dtype: object


In [ ]:
# Check row wise- is this entire row an identical copy of another row?
dupes = df.duplicated().sum()
print(f'Duplicate rows found: {dupes:,}')

df = df.drop_duplicates()
print(f'Rows remaining: {len(df):,}')

Duplicate rows found: 26,124
Rows remaining: 779,425


In [ ]:
# revenue = quantity x price → most commonly queried metric in retail analytics
# adding it now means SQL queries can use it directly without recalculating
# .round(2) → keep to 2 decimal places (pence/cents)
df['revenue'] = (df['quantity'] * df['price']).round(2)

print(f'Revenue column added.')
print(f'Total revenue in clean dataset: £{df["revenue"].sum():,.2f}')
df[['invoice', 'customer_id', 'quantity', 'price', 'revenue']].head()


Revenue column added.
Total revenue in clean dataset: £17,374,804.25


,invoice,customer_id,quantity,price,revenue
0,489434,13085,12,6.95,83.4
1,489434,13085,12,6.75,81.0
2,489434,13085,12,6.75,81.0
3,489434,13085,48,2.10,100.8
4,489434,13085,24,1.25,30.0


In [ ]:
# final_count = current size of df after all cleaning steps
# removed = original_count (Cell 6) - final_count → total rows dropped
# this summary documents every decision made goes straight into the README
# >10 in formatting → right aligns numbers so output looks like a proper report
final_count  = len(df)
removed      = original_count - final_count

print('============================================')
print('           CLEANING SUMMARY')
print('============================================')
print(f'  Original rows       : {original_count:>10,}')
print(f'  Rows removed        : {removed:>10,}  ({removed/original_count*100:.1f}%)')
print(f'  Final clean rows    : {final_count:>10,}')
print(f'  Columns             : {df.shape[1]}')
print(f'  Date range          : {df["invoicedate"].min().date()} to {df["invoicedate"].max().date()}')
print(f'  Unique customers    : {df["customer_id"].nunique():,}')
print(f'  Unique products     : {df["stockcode"].nunique():,}')
print(f'  Countries           : {df["country"].nunique()}')
print(f'  Total revenue       : £{df["revenue"].sum():,.2f}')
print('============================================')

           CLEANING SUMMARY
  Original rows       :  1,067,371
  Rows removed        :    287,946  (27.0%)
  Final clean rows    :    779,425
  Columns             : 9
  Date range          : 2009-12-01 to 2011-12-09
  Unique customers    : 5,878
  Unique products     : 4,631
  Countries           : 41
  Total revenue       : £17,374,804.25


In [ ]:
# Export the cleaned dataframe as a new CSV file back to Google Drive
# This is the file we load into MySQL Workbench for all SQL analysis
# index=False → don't write the row numbers into the file, they're not real data
OUTPUT_PATH = '/content/drive/My Drive/CustomerBehaviourProject/Dataset/online_retail_clean.csv'

df.to_csv(OUTPUT_PATH, index=False)

print(f'Exported clean file to: {OUTPUT_PATH}')
print(f'Final shape: {df.shape}')

Exported clean file to: /content/drive/My Drive/CustomerBehaviourProject/Dataset/online_retail_clean.csv
Final shape: (779425, 9)
